# Capstone 3 — Enterprise Knowledge Assistant

You will ship a RAG chatbot grounded in a **real corpus** of your choice (your company wiki, the LMS course content, a research-paper collection — at least 50 documents). It must produce **citations**, **refuse** when context is insufficient, and survive a graded retrieval + faithfulness eval.

## Phase 0 — Concept Recall

1. Why is "Recall@k" measured separately from "answer faithfulness"?
2. When would you choose **MMR** over plain top-k cosine?
3. Re-ranking adds latency. Under what concrete condition is the latency worth it?

*(5 pts each.)*

In [ ]:
# Phase 1 — Setup
# pip install langchain langchain-community langchain-openai chromadb sentence-transformers rank-bm25 pypdf

import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.retrievers import BM25Retriever, EnsembleRetriever

CORPUS_DIR = "./corpus"   # TODO 1: drop ≥50 PDFs (or .md files) here


In [ ]:
# Phase 2 — Ingestion (Guided Build)
loader = PyPDFDirectoryLoader(CORPUS_DIR)
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
chunks = splitter.split_documents(docs)
print(f"{len(docs)} docs → {len(chunks)} chunks")

# TODO 2: enrich each chunk's metadata with {source, page, ingested_at}
# TODO 3: drop chunks shorter than 50 chars (boilerplate)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vec = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma")


In [ ]:
# Phase 3 — Hybrid retriever + reranker
bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 10
dense = vec.as_retriever(search_kwargs={"k": 10})

hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.3, 0.7])

# TODO 4 (extension): wrap hybrid with a reranker
# from langchain.retrievers import ContextualCompressionRetriever
# from langchain_cohere import CohereRerank
# reranker = CohereRerank(model="rerank-english-v3.0", top_n=5)
# retriever = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=hybrid)
retriever = hybrid


In [ ]:
# Phase 4 — Generation with citations + refusal

from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

SYSTEM = """You are a precise assistant. Answer ONLY from the supplied context.
- Every claim must cite the source like [source.pdf p.3].
- If the context is insufficient, reply exactly: 'I do not have enough information to answer that.'
- Do not use outside knowledge."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

def format_ctx(docs):
    return "\n\n".join(f"[{d.metadata.get('source','?')} p.{d.metadata.get('page','?')}] {d.page_content}" for d in docs)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag = (
    {"context": retriever | format_ctx, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print(rag.invoke("What does the corpus say about X?"))


In [ ]:
# Phase 5 — Eval suite (this is what you are graded on)

EVAL = [
    # {"q": "...", "must_cite": ["doc_a.pdf"], "answer_substr": "..."},
    # TODO 5: hand-write 30+ questions covering all parts of the corpus,
    # plus 5 OUT-OF-CORPUS questions where the model must refuse.
]

def recall_at_k(retriever, q, gold_sources, k=5):
    docs = retriever.get_relevant_documents(q)[:k]
    sources = {d.metadata.get("source") for d in docs}
    return len(sources & set(gold_sources)) / len(gold_sources)

def faithful(answer, must_cite):
    return all(c in answer for c in must_cite)

def refused(answer):
    return "do not have enough information" in answer.lower()

# TODO 6: run all three metrics across EVAL, print aggregates.


In [ ]:
# Phase 6 — Monitoring

import time, json, logging
log = logging.getLogger("rag")

def ask(q):
    t = {}
    t["start"] = time.time()
    docs = retriever.get_relevant_documents(q)
    t["retrieved"] = time.time()
    answer = rag.invoke(q)
    t["answered"] = time.time()
    log.info(json.dumps({
        "question": q,
        "n_docs": len(docs),
        "retrieve_ms": int((t["retrieved"]-t["start"])*1000),
        "generate_ms": int((t["answered"]-t["retrieved"])*1000),
        "refused": "do not have enough" in answer.lower(),
        # TODO 7: add token / cost estimates
    }))
    return answer


## Phase 7 — Submission checklist (rubric / 100)

- [ ] **20 pts** — Recall@5 ≥ 0.75 on the in-corpus questions.
- [ ] **25 pts** — Answer faithfulness ≥ 0.85 (judged by LLM-as-judge or manually).
- [ ] **15 pts** — All cited pages exist and contain the cited claim (spot-check 10).
- [ ] **10 pts** — Refusal precision ≥ 0.9 on out-of-corpus questions.
- [ ] **15 pts** — p95 end-to-end latency < 3 s.
- [ ] **10 pts** — Cost per query reported and < $0.01 for `gpt-4o-mini`-class models.
- [ ] **5 pts** — Re-indexing pipeline is one command and embeddings are versioned.

### What you will have *learned by doing*
- Why "RAG works on the demo" is not "RAG works in production".
- That hybrid retrieval beats pure dense for messy real-world queries.
- That refusal is a feature, not a failure.